# Interpretability deep-dive — K=5 rain-prediction XGBoost

A white-box characterization of the deployed model (base+temporal+spatial, $K=5$, frozen EXP-002
config). Le modèle est réentraîné ici sur la période de train **du pipeline**
(`weatherAUS_X_train.csv` / `weatherAUS_y_train.csv`, augmentées des blocs temporel et spatial
construits sur `weatherAUS_trend_grouped.csv`) et expliqué sur le test 2016+, donc les
attributions décrivent le comportement sur des données non vues à l'entraînement.

**Toolkit.** The analysis is organized around four complementary techniques, each answering a
question the others cannot:

- **SHAP** (`TreeExplainer`) — exact per-prediction Shapley attributions. Global structure
  (beeswarm, block attribution), **pairwise interaction values** (the humidity×pressure×sunshine
  conjunctions that justified trees over the linear baseline), and local waterfalls for individual
  forecasts.
- **Permutation importance** (scikit-learn) — model-agnostic, test-set importance; triangulated
  against SHAP and gain, which are computed differently and can disagree.
- **PDP + ICE, and ALE** (scikit-learn / PyALE) — marginal-effect curves. ALE is included because
  the features are strongly correlated (the pressure pair, and each spatial `nbr_*` with its home
  twin), a regime in which PDP is biased and ALE is the honest estimator.
- **SHAP-based feature clustering** — visualizes the redundancy the audit and antecedent
  experiments inferred indirectly.

Logs -> `INTERP_result.txt`, figures `interp_*`, modèle réentraîné sérialisé dans `../../models/`.


## 1. Setup

In [ ]:
# --- Environnement ---
# Ce notebook tourne en local sur les sorties du pipeline de preprocessing
# (dossier ../../data). Sur Colab, installer les dépendances puis pointer
# WEATHER_DATA_DIR vers le dossier contenant les CSV du pipeline.
import subprocess, sys as _sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run([_sys.executable, '-m', 'pip', '-q', 'install', '-U', 'xgboost', 'shap', 'PyALE'],
                   check=False)
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                    '--format=csv,noheader'], check=False)

print(f'Running on Colab: {IN_COLAB}')


In [ ]:
import os, sys, zipfile, warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
try:
    from PyALE import ale as pyale
    HAS_ALE = True
except Exception:
    HAS_ALE = False

RANDOM_STATE = 42
TEST_CUTOFF = pd.Timestamp('2016-01-01')
K_NEIGHBOURS = 5

def _gpu_available():
    try:
        d = xgb.DMatrix(np.zeros((8, 2)), label=np.zeros(8))
        xgb.train({'device': 'cuda', 'tree_method': 'hist'}, d, num_boost_round=1)
        return True
    except Exception:
        return False
USE_GPU = _gpu_available()
DEVICE = 'cuda' if USE_GPU else 'cpu'
V2_PARAMS = dict(max_depth=6, min_child_weight=10, gamma=2.0, reg_lambda=20, reg_alpha=0.5,
                 learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, n_estimators=800)
print(f'xgboost {xgb.__version__} | device {DEVICE} | shap {HAS_SHAP} | PyALE {HAS_ALE}')

### Output setup

In [ ]:
os.makedirs('figures', exist_ok=True)
_PATH = 'INTERP_result.txt'
_FILE = open(_PATH, 'w', encoding='utf-8')
class _Tee:
    def __init__(self, *s): self._s = s
    def write(self, d):
        for s in self._s: s.write(d)
    def flush(self):
        for s in self._s: s.flush()
    def isatty(self): return False
_ORIG = sys.stdout; sys.stdout = _Tee(_ORIG, _FILE)
print('# Interpretability deep-dive (K=5) — numerical outputs')
print(f'# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
def log_section(t):
    print('\n' + '=' * 72 + '\n' + t + '\n' + '=' * 72)
def save_fig(name, fig=None):
    fig = fig or plt.gcf(); p = f'figures/{name}.png'
    fig.savefig(p, dpi=150, bbox_inches='tight'); print(f'  [saved {p}]')

# --- Sérialisation des modèles ---------------------------------------------
# Chaque étape du notebook est enregistrée dans <repo>/models sous forme d'un
# dict joblib : le modèle + les métadonnées nécessaires pour le rejouer.
MODEL_DIR = os.environ.get('WEATHER_MODEL_DIR', '../../models')
os.makedirs(MODEL_DIR, exist_ok=True)
NOTEBOOK = 'interpretability_XGB'

def save_model(obj, name, **meta):
    """Sauvegarde `obj` dans MODEL_DIR/<name>.joblib avec ses métadonnées."""
    path = f'{MODEL_DIR}/{name}.joblib'
    joblib.dump({'model': obj,
                 'notebook': NOTEBOOK,
                 'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
                 **meta}, path)
    print(f'  [saved {path} ({os.path.getsize(path) / 1e6:.1f} MB)]')


## 2. Rebuild features from the pipeline (base + temporal + spatial K=5) and refit

Mêmes briques que `EXP-007_XGBoost_deploy.ipynb` : base et split depuis les matrices du pipeline,
lags et voisins depuis le frame pré-imputation.


In [ ]:
log_section('2. Data loading (sorties du pipeline)')

# Sorties du pipeline, dans <repo>/data :
#   3.preprocessing_trend_group.ipynb -> weatherAUS_trend_grouped.csv
#   4_preprocessing_imputation.ipynb  -> weatherAUS_MICE_cyclic.csv
#                                        weatherAUS_X_train.csv / weatherAUS_y_train.csv
#                                        weatherAUS_X_test.csv  / weatherAUS_y_test.csv
DATA_DIR = os.environ.get('WEATHER_DATA_DIR', '../../data')

# Matrices déjà splitées, encodées (LabelEncoder) et scalées (RobustScaler ajusté
# sur le train uniquement) par 4_preprocessing_imputation.ipynb.
X_train = pd.read_csv(f'{DATA_DIR}/weatherAUS_X_train.csv')
X_test  = pd.read_csv(f'{DATA_DIR}/weatherAUS_X_test.csv')
y_train = pd.read_csv(f'{DATA_DIR}/weatherAUS_y_train.csv').squeeze('columns').astype(int)
y_test  = pd.read_csv(f'{DATA_DIR}/weatherAUS_y_test.csv').squeeze('columns').astype(int)

# Index (Date, Location) aligné ligne à ligne sur X_train / X_test :
# weatherAUS_MICE_cyclic.csv est trié par Date et le pipeline y applique le même
# seuil TEST_CUTOFF, donc ses len(X_train) premières lignes SONT celles de X_train.
meta = pd.read_csv(f'{DATA_DIR}/weatherAUS_MICE_cyclic.csv',
                   usecols=['Date', 'Location', 'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm'], parse_dates=['Date'])
meta = meta[['Date', 'Location', 'Pressure9am', 'Pressure3pm', 'Temp9am', 'Temp3pm']]
assert len(meta) == len(X_train) + len(X_test), 'X_train/X_test ne recouvrent pas MICE_cyclic'
meta_train = meta.iloc[:len(X_train)].reset_index(drop=True)
meta_test  = meta.iloc[len(X_train):].reset_index(drop=True)
assert (meta_train['Date'] < TEST_CUTOFF).all(), 'lignes de train postérieures au seuil'
assert (meta_test['Date'] >= TEST_CUTOFF).all(), 'lignes de test antérieures au seuil'

print(f'X_train {X_train.shape} | X_test {X_test.shape}')
print(f'Train {meta_train["Date"].min().date()} -> {meta_train["Date"].max().date()} | '
      f'Test {meta_test["Date"].min().date()} -> {meta_test["Date"].max().date()}')
print(f'Stations : {meta["Location"].nunique()} | pluie train {y_train.mean():.4%} / test {y_test.mean():.4%}')

# Frame AVANT imputation (mêmes lignes, NaN d'origine conservés) : sert à
# reconstruire les features retardées / voisines sans réutiliser de valeur
# reconstruite par MICE.
pre = pd.read_csv(f'{DATA_DIR}/weatherAUS_trend_grouped.csv', parse_dates=['Date'])
assert len(pre) == len(meta), 'weatherAUS_trend_grouped.csv désaligné du pipeline'
print(f'Pré-imputation : {pre.shape[0]:,} lignes x {pre.shape[1]} colonnes '
      f'({pre.isna().sum().sum():,} NaN conservés)')

# --- Features de base (v2) ---
# Features de tendance intra-journalière. Elles sont calculées sur les valeurs
# NON scalées de weatherAUS_MICE_cyclic.csv : une différence de colonnes déjà
# passées au RobustScaler (échelles distinctes) ne serait pas la même grandeur.
for X, m in ((X_train, meta_train), (X_test, meta_test)):
    X['Pressure_diff'] = (m['Pressure3pm'] - m['Pressure9am']).to_numpy()
    X['Temp_diff']     = (m['Temp3pm']     - m['Temp9am']).to_numpy()

# Le pipeline livre 29 colonnes. On retire les mêmes que l'étude LR pour que les
# expériences EXP-001..007 restent comparables entre elles :
DROP_COLS = [
    'WindGustDir', 'WindDir9am', 'WindDir3pm',  # labels encodés ; les sin/cos sont conservés
    'RainToday',                                # quasi-fuite (on donne déjà le reste de la journée)
    'Humidity9am',                              # on garde Humidity3pm (plus prédictive)
    'Month',                                    # redondant avec Date_sin / Date_cos
]
BASE_FEATURES = [c for c in X_train.columns if c not in DROP_COLS]

# Matrice complète dans l'ordre chronologique du pipeline : X_train puis X_test.
# Les blocs de features additionnels (temporel / spatial) viendront s'y coller.
X_all = pd.concat([X_train[BASE_FEATURES], X_test[BASE_FEATURES]], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)
loc_all = meta['Location']
dates   = meta['Date']
N_TRAIN = len(X_train)

def align_to_pipeline(frame, cols):
    """Réaligne un bloc de features indexé par (Date, Location) sur l'ordre des
    lignes du pipeline. Les deux frames partagent la même clé unique, donc le
    réalignement est exact et ne peut pas dupliquer de ligne."""
    key = pd.MultiIndex.from_frame(frame[['Date', 'Location']])
    assert key.is_unique, 'clé (Date, Location) non unique dans le bloc de features'
    out = frame.set_axis(key)[cols].reindex(pd.MultiIndex.from_frame(meta[['Date', 'Location']]))
    return out.reset_index(drop=True)

print(f'BASE features: {len(BASE_FEATURES)} | X_all {X_all.shape}')

# --- Features temporelles (frame pré-imputation) ---
# Toutes les features ci-dessous n'utilisent que les jours <= t-1 (pas d'info du jour
# même, pas de futur), construites par `Location` sur weatherAUS_trend_grouped.csv,
# c'est-à-dire AVANT l'imputation MICE : aucune valeur reconstruite ne peut remonter
# dans un lag. Les NaN qui en résultent (premiers jours / manquants d'origine) sont
# laissés tels quels — XGBoost les gère nativement et EXP-003 a montré que les
# imputer n'apporte rien.
pre_s = pre.sort_values(['Location', 'Date']).copy()
g = pre_s.groupby('Location', sort=False)

LAG_COLS = ['Humidity3pm','Pressure3pm','Rainfall','Cloud3pm','Sunshine','Temp3pm','WindGustSpeed']
feat = pd.DataFrame({'Date': pre_s['Date'].values, 'Location': pre_s['Location'].values})
for c in LAG_COLS:
    feat[f'{c}_lag1'] = g[c].shift(1).values

# multi-day, all strictly past (>= t-1)
feat['Pressure_3d_chg']  = (g['Pressure3pm'].shift(1) - g['Pressure3pm'].shift(4)).values
feat['Humidity_3d_mean'] = g['Humidity3pm'].apply(lambda s: s.shift(1).rolling(3).mean()).values
feat['Rainfall_3d_sum']  = g['Rainfall'].apply(lambda s: s.shift(1).rolling(3).sum()).values

# yesterday's rain flag + days since last rain (computed on the lagged flag -> strictly past)
rt = pre_s['RainToday'].astype(int)          # booléen dans le frame du pipeline
rt_lag = rt.groupby(pre_s['Location'], sort=False).shift(1)
feat['RainToday_lag1'] = rt_lag.values
def _days_since(s):
    out = np.empty(len(s)); cnt = 9999.0
    for i, v in enumerate(s.to_numpy()):
        out[i] = cnt
        cnt = 0.0 if v == 1 else (cnt + 1 if cnt < 9999 else 9999)
    return out
feat['days_since_rain'] = (
    rt_lag.fillna(0).groupby(pre_s['Location'], sort=False).transform(lambda s: _days_since(s)).values)

TEMPORAL_FEATURES = [c for c in feat.columns if c not in ('Date', 'Location')]
X_all[TEMPORAL_FEATURES] = align_to_pipeline(feat, TEMPORAL_FEATURES)
nan_share = X_all[TEMPORAL_FEATURES].isna().mean().mean()
print(f'Temporal features: {len(TEMPORAL_FEATURES)} -> {TEMPORAL_FEATURES}')
print(f'Mean NaN share across temporal features: {nan_share:.1%} (left for XGBoost native handling)')

# --- Features spatiales (K=5), frame pré-imputation -------------------------
COORDS = {
    'Adelaide': (-34.9285, 138.6007), 'Albany': (-35.0269, 117.8837), 'Albury': (-36.0737, 146.9135),
    'AliceSprings': (-23.6980, 133.8807), 'BadgerysCreek': (-33.8829, 150.7609), 'Ballarat': (-37.5622, 143.8503),
    'Bendigo': (-36.7570, 144.2794), 'Brisbane': (-27.4698, 153.0251), 'Cairns': (-16.9186, 145.7781),
    'Canberra': (-35.2809, 149.1300), 'Cobar': (-31.4958, 145.8389), 'CoffsHarbour': (-30.2986, 153.1094),
    'Dartmoor': (-37.9225, 141.2767), 'Darwin': (-12.4634, 130.8456), 'GoldCoast': (-28.0167, 153.4000),
    'Hobart': (-42.8821, 147.3272), 'Katherine': (-14.4652, 132.2635), 'Launceston': (-41.4332, 147.1441),
    'Melbourne': (-37.8136, 144.9631), 'MelbourneAirport': (-37.6690, 144.8410), 'Mildura': (-34.1855, 142.1625),
    'Moree': (-29.4639, 149.8407), 'MountGambier': (-37.8284, 140.7820), 'MountGinini': (-35.5294, 148.7723),
    'Newcastle': (-32.9283, 151.7817), 'Nhil': (-36.3333, 141.6500), 'NorahHead': (-33.2833, 151.5667),
    'NorfolkIsland': (-29.0408, 167.9547), 'Nuriootpa': (-34.4700, 138.9960), 'PearceRAAF': (-31.6676, 116.0292),
    'Penrith': (-33.7507, 150.6877), 'Perth': (-31.9505, 115.8605), 'PerthAirport': (-31.9385, 115.9672),
    'Portland': (-38.3609, 141.6041), 'Richmond': (-33.6000, 150.7667), 'Sale': (-38.1026, 147.0730),
    'SalmonGums': (-32.9815, 121.6438), 'Sydney': (-33.8688, 151.2093), 'SydneyAirport': (-33.9399, 151.1753),
    'Townsville': (-19.2590, 146.8169), 'Tuggeranong': (-35.4244, 149.0888), 'Uluru': (-25.3444, 131.0369),
    'WaggaWagga': (-35.1082, 147.3598), 'Walpole': (-34.9777, 116.7338), 'Watsonia': (-37.7100, 145.0830),
    'Williamtown': (-32.8150, 151.8428), 'Witchcliffe': (-34.0261, 115.1003), 'Wollongong': (-34.4278, 150.8931),
    'Woomera': (-31.1656, 136.8193),
}
assert set(meta['Location'].unique()) <= set(COORDS)
stations = sorted(COORDS)
def haversine(a, b):
    R=6371.0; p1,p2=np.radians(a[0]),np.radians(b[0]); dphi=np.radians(b[0]-a[0]); dl=np.radians(b[1]-a[1])
    return 2*R*np.arcsin(np.sqrt(np.sin(dphi/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2))
NEIGHBOURS, NEAREST_DIST = {}, {}
for s in stations:
    dd = sorted((haversine(COORDS[s], COORDS[t]), t) for t in stations if t != s)
    NEIGHBOURS[s] = [t for _, t in dd[:K_NEIGHBOURS]]; NEAREST_DIST[s] = dd[0][0]
pre2 = pre.assign(_RainToday=pre['RainToday'].astype(int))
full_dates = pd.date_range(pre['Date'].min(), pre['Date'].max(), freq='D')
WIDE = {v: (pre2.pivot_table(index='Date', columns='Location', values=v, dropna=False)
           .reindex(index=full_dates, columns=stations))
        for v in ['Rainfall','Pressure3pm','Humidity3pm','_RainToday']}
SPEC = [('Rainfall',False),('Rainfall',True),('Pressure3pm',False),('Pressure3pm',True),
        ('Humidity3pm',False),('_RainToday',False)]
sfeat = meta[['Date','Location']].copy(); SPATIAL_FEATURES = []
for var, lag in SPEC:
    agg = pd.DataFrame({s: WIDE[var][NEIGHBOURS[s]].mean(axis=1) for s in stations}, index=WIDE[var].index)
    if lag: agg = agg.shift(1)
    agg.index.name = 'Date'
    name = f'nbr_{var.replace("_RainToday","RainToday")}_{"t1" if lag else "t"}'
    sfeat = sfeat.merge(agg.reset_index().melt(id_vars='Date', var_name='Location', value_name=name),
                        on=['Date','Location'], how='left')
    SPATIAL_FEATURES.append(name)
sfeat['nearest_dist'] = sfeat['Location'].map(NEAREST_DIST); SPATIAL_FEATURES.append('nearest_dist')
X_all[SPATIAL_FEATURES] = align_to_pipeline(sfeat, SPATIAL_FEATURES)

# --- Split du pipeline + refit ----------------------------------------------
FEATURES = BASE_FEATURES + TEMPORAL_FEATURES + SPATIAL_FEATURES
assert list(X_all.columns) == FEATURES
BLOCK = {**{f:'base' for f in BASE_FEATURES}, **{f:'temporal' for f in TEMPORAL_FEATURES},
         **{f:'spatial' for f in SPATIAL_FEATURES}}
X_train = X_all.iloc[:N_TRAIN].reset_index(drop=True)
X_test  = X_all.iloc[N_TRAIN:].reset_index(drop=True)
y_train = y_all.iloc[:N_TRAIN].reset_index(drop=True)
y_test  = y_all.iloc[N_TRAIN:].reset_index(drop=True)
SPW = (1 - y_train.mean()) / y_train.mean()

model = XGBClassifier(tree_method='hist', device=DEVICE, eval_metric='logloss',
                      random_state=RANDOM_STATE, n_jobs=-1, scale_pos_weight=SPW, **V2_PARAMS)
model.fit(X_train, y_train)
model.set_params(device='cpu')   # SHAP/permutation on CPU
from sklearn.metrics import f1_score, roc_auc_score
proba = model.predict_proba(X_test)[:, 1]
print(f'{len(FEATURES)} features | train {len(X_train):,} test {len(X_test):,}')
print(f'sanity: test ROC-AUC {roc_auc_score(y_test, proba):.4f}')
save_model(model, 'interpretability_01_refit_K5',
           features=FEATURES, blocks=BLOCK, stage='refit K=5 expliqué dans ce notebook',
           params=V2_PARAMS, scale_pos_weight=float(SPW), K_neighbours=K_NEIGHBOURS,
           test_roc_auc=float(roc_auc_score(y_test, proba)))


## 3. SHAP attribution on the test set

`TreeExplainer` computes exact Shapley values in the model's log-odds (margin) space. We explain a
test-set sample; positive SHAP pushes toward rain.

In [ ]:
log_section('3. SHAP values')
if not HAS_SHAP:
    raise RuntimeError('shap unavailable — install it to run this notebook')
N_SHAP = min(6000, len(X_test))
rng = np.random.RandomState(RANDOM_STATE)
idx = rng.choice(len(X_test), N_SHAP, replace=False)
X_s = X_test.iloc[idx].reset_index(drop=True)
y_s = y_test.iloc[idx].reset_index(drop=True)
proba_s = proba[idx]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_s)          # (N_SHAP, n_features), positive-class margin
base_value = float(np.ravel(explainer.expected_value)[0])
mean_abs = pd.DataFrame({'feature': FEATURES, 'mean_abs_SHAP': np.abs(shap_values).mean(0),
                         'block': [BLOCK[f] for f in FEATURES]}).sort_values('mean_abs_SHAP', ascending=False)
print(f'Explained {N_SHAP} test rows | base margin {base_value:.3f}')
print('\nTop 15 by mean |SHAP|:')
print(mean_abs.head(15).to_string(index=False, float_format='%.4f'))

### 3.1 Global structure — beeswarm and bar

In [ ]:
fig = plt.figure()
shap.summary_plot(shap_values, X_s, max_display=20, show=False)
save_fig('interp_01_beeswarm', plt.gcf()); plt.show()

fig, ax = plt.subplots(figsize=(8, 7))
top = mean_abs.head(20)
cmap = {'base':'steelblue','temporal':'darkorange','spatial':'crimson'}
ax.barh(top['feature'][::-1], top['mean_abs_SHAP'][::-1], color=[cmap[b] for b in top['block']][::-1])
ax.set_xlabel('mean |SHAP| (log-odds)'); ax.set_title('Global importance by block — base/temporal/spatial')
ax.grid(axis='x', ls='--', alpha=0.4); plt.tight_layout()
save_fig('interp_02_meanshap_bar', fig); plt.show()

### 3.2 Importance triangulation — gain vs. permutation vs. SHAP

The three measures answer different questions: gain is a train-time split statistic (biased toward
high-cardinality features), permutation importance is a test-time drop in PR-AUC when a feature is
shuffled (model-agnostic, correlation-sensitive), and mean $|\text{SHAP}|$ is the average
attribution magnitude. Agreement across all three is strong evidence a feature carries genuine,
non-redundant signal; disagreement localizes redundancy.

In [ ]:
log_section('3.2 Importance triangulation')
gain = model.get_booster().get_score(importance_type='gain')
gain_s = pd.Series({f: gain.get(f, 0.0) for f in FEATURES})

n_perm = min(8000, len(X_test))
pidx = rng.choice(len(X_test), n_perm, replace=False)
perm = permutation_importance(model, X_test.iloc[pidx], y_test.iloc[pidx],
                              scoring='average_precision', n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
tri = pd.DataFrame({'feature': FEATURES, 'block':[BLOCK[f] for f in FEATURES],
                    'gain': gain_s.values,
                    'perm_dPR': perm.importances_mean,
                    'mean_abs_SHAP': np.abs(shap_values).mean(0)})
for col in ['gain','perm_dPR','mean_abs_SHAP']:
    tri[col+'_rank'] = tri[col].rank(ascending=False).astype(int)
tri = tri.sort_values('mean_abs_SHAP', ascending=False)
print('Rank by each measure (top 15 by SHAP):')
print(tri.head(15)[['feature','block','gain_rank','perm_dPR_rank','mean_abs_SHAP_rank']].to_string(index=False))

fig, ax = plt.subplots(figsize=(6.5, 6.5))
sc = ax.scatter(tri['perm_dPR_rank'], tri['mean_abs_SHAP_rank'],
                c=[cmap[b] for b in tri['block']], s=30)
for _, r in tri.head(10).iterrows():
    ax.annotate(r['feature'], (r['perm_dPR_rank'], r['mean_abs_SHAP_rank']), fontsize=7)
ax.plot([1,len(FEATURES)],[1,len(FEATURES)],'k:',alpha=0.4)
ax.set_xlabel('permutation-importance rank'); ax.set_ylabel('mean |SHAP| rank')
ax.set_title('Triangulation: permutation vs SHAP rank (diagonal = agreement)')
ax.invert_xaxis(); ax.invert_yaxis(); ax.grid(alpha=0.3); plt.tight_layout()
save_fig('interp_03_triangulation', fig); plt.show()

### 3.3 Feature-block attribution

Summing mean $|\text{SHAP}|$ within each engineered block quantifies how the model's attribution
budget is distributed — the SHAP-space analogue of the ablation ladder in the report.

In [ ]:
log_section('3.3 Block attribution')
block_share = mean_abs.groupby('block')['mean_abs_SHAP'].sum()
block_share = (block_share / block_share.sum() * 100).round(1).sort_values(ascending=False)
print('Share of total mean|SHAP| by block (%):')
print(block_share.to_string())
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(block_share.index, block_share.values, color=[cmap[b] for b in block_share.index])
for i,v in enumerate(block_share.values): ax.text(i, v+0.5, f'{v:.1f}%', ha='center')
ax.set_ylabel('% of total mean|SHAP|'); ax.set_title('Attribution budget by feature block')
plt.tight_layout(); save_fig('interp_04_block_attribution', fig); plt.show()

## 4. Dependence and physical plausibility

SHAP dependence (scatter) plots show, for one feature, how its attribution varies with its value —
the model's learned response curve. Physically implausible shapes (e.g. non-monotone humidity)
would be a trust red flag; monotone, sign-correct curves build confidence.

In [ ]:
log_section('4. SHAP dependence')
for f in ['Humidity3pm','Pressure3pm','Sunshine','days_since_rain','nbr_Humidity3pm_t','Pressure_diff']:
    if f not in FEATURES: continue
    try:
        fig = plt.figure()
        shap.dependence_plot(f, shap_values, X_s, interaction_index='auto', show=False)
        save_fig(f'interp_05_dep_{f}', plt.gcf()); plt.show()
    except Exception as e:
        print(f'dependence {f} skipped ({type(e).__name__})')

## 5. Interaction values — the conjunctions that justify trees

The linear baseline could not represent joint conditions such as "high humidity **and** falling
pressure". SHAP interaction values decompose each prediction into main effects plus pairwise
interaction terms; the mean absolute off-diagonal entries rank which feature *pairs* the model
treats non-additively. A non-trivial interaction structure here is the direct evidence that the
tree's advantage over the linear score is real and localizable.

In [ ]:
log_section('5. SHAP interaction values')
try:
    N_INT = min(2000, len(X_s))
    X_i = X_s.iloc[:N_INT]
    inter = explainer.shap_interaction_values(X_i)         # (N_INT, F, F)
    M = np.abs(inter).mean(0)
    np.fill_diagonal(M, 0.0)                                # keep interactions only
    Mdf = pd.DataFrame(M, index=FEATURES, columns=FEATURES)
    pairs = []
    for a in range(len(FEATURES)):
        for b in range(a+1, len(FEATURES)):
            pairs.append((FEATURES[a], FEATURES[b], M[a, b]))
    pairs = pd.DataFrame(pairs, columns=['f1','f2','mean_abs_interaction']).sort_values('mean_abs_interaction', ascending=False)
    print('Top 15 feature-pair interactions:')
    print(pairs.head(15).to_string(index=False, float_format='%.4f'))

    top_feats = mean_abs.head(12)['feature'].tolist()
    sub = Mdf.loc[top_feats, top_feats]
    fig, ax = plt.subplots(figsize=(9, 7.5))
    im = ax.imshow(sub.values, cmap='magma')
    ax.set_xticks(range(len(top_feats))); ax.set_xticklabels(top_feats, rotation=90, fontsize=8)
    ax.set_yticks(range(len(top_feats))); ax.set_yticklabels(top_feats, fontsize=8)
    ax.set_title('Mean |SHAP interaction| (top-12 features)')
    fig.colorbar(im, fraction=0.046); plt.tight_layout()
    save_fig('interp_06_interaction_heatmap', fig); plt.show()
except Exception as e:
    print(f'interaction values skipped ({type(e).__name__}: {e})')

## 6. Marginal effects — PDP/ICE and ALE under feature correlation

Partial Dependence (PDP) marginalizes a feature's effect over the data distribution; Individual
Conditional Expectation (ICE) shows the per-sample curves PDP averages, exposing heterogeneity.
Both assume the feature can be varied independently — false here, since (for example)
`nbr_Humidity3pm_t` is strongly correlated with the home `Humidity3pm`. Accumulated Local Effects
(ALE) conditions on the local feature distribution and is unbiased under such correlation;
contrasting PDP and ALE on a correlated feature quantifies the PDP bias.

In [ ]:
log_section('6. PDP / ICE')
X_pdp = X_s.iloc[:2000]
try:
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    PartialDependenceDisplay.from_estimator(model, X_pdp, ['Humidity3pm'], kind='both',
                                            ax=ax[0], ice_lines_kw={'alpha':0.1})
    ax[0].set_title('PDP + ICE — Humidity3pm')
    PartialDependenceDisplay.from_estimator(model, X_pdp, ['Pressure_diff'], kind='both',
                                            ax=ax[1], ice_lines_kw={'alpha':0.1})
    ax[1].set_title('PDP + ICE — Pressure_diff')
    plt.tight_layout(); save_fig('interp_07_pdp_ice', fig); plt.show()
except Exception as e:
    print(f'PDP skipped ({type(e).__name__}: {e})')

In [ ]:
# ALE for a correlated spatial feature, contrasted with its PDP
if HAS_ALE:
    class _ProbaModel:
        def __init__(self, m): self.m = m
        def predict(self, X): return self.m.predict_proba(X)[:, 1]
    try:
        wrapped = _ProbaModel(model)
        fig, ax = plt.subplots(figsize=(7, 5))
        pyale(X=X_s.iloc[:5000], model=wrapped, feature=['nbr_Humidity3pm_t'],
              grid_size=40, include_CI=True, fig=fig, ax=ax)
        ax.set_title('ALE — nbr_Humidity3pm_t (correlation-robust)')
        plt.tight_layout(); save_fig('interp_08_ale_nbr_humidity', fig); plt.show()
    except Exception as e:
        print(f'ALE skipped ({type(e).__name__}: {e})')
else:
    print('PyALE unavailable — PDP/ICE above; install PyALE for correlation-robust ALE.')

## 7. Redundancy — SHAP-based feature clustering

The audit and antecedent experiments inferred redundancy indirectly (features used but adding
$\approx 0$). Hierarchical clustering on the features, with the bar plot ordered by that
clustering, makes the redundant groups explicit: tightly-joined leaves are features the model
treats as substitutes.

In [ ]:
log_section('7. Redundancy clustering')
try:
    clustering = shap.utils.hclust(X_s, y_s)
    expl = shap.Explanation(values=shap_values, base_values=np.full(len(X_s), base_value),
                            data=X_s.values, feature_names=FEATURES)
    fig = plt.figure()
    shap.plots.bar(expl, clustering=clustering, clustering_cutoff=0.5, max_display=20, show=False)
    save_fig('interp_09_redundancy_clustering', plt.gcf()); plt.show()
except Exception as e:
    print(f'clustering skipped ({type(e).__name__}: {e})')

## 8. Local explanations — three forecast archetypes

Waterfall plots decompose a single prediction from the base rate to the final margin. We inspect a
confident correct rain call, a false alarm (predicted rain, none fell), and a missed event
(predicted dry, rain fell), to read *why* each decision was made and what distinguishes the errors.

In [ ]:
log_section('8. Local waterfalls')
thr = 0.60
pred_s = (proba_s >= thr).astype(int)
def pick(cond):
    w = np.where(cond)[0]
    return int(w[0]) if len(w) else None
cases = {'confident_rain': pick((y_s.values==1) & (proba_s>0.8)),
         'false_alarm':    pick((y_s.values==0) & (pred_s==1) & (proba_s>0.65)),
         'missed_event':   pick((y_s.values==1) & (pred_s==0) & (proba_s<0.30))}
for label, i in cases.items():
    if i is None:
        print(f'{label}: no matching case'); continue
    print(f'{label}: idx {i} | p_rain {proba_s[i]:.3f} | actual {int(y_s.values[i])} '
          f'| {meta_test.iloc[idx[i]]["Location"]} {meta_test.iloc[idx[i]]["Date"].date()}')
    try:
        e = shap.Explanation(values=shap_values[i], base_values=base_value,
                             data=X_s.iloc[i], feature_names=FEATURES)
        fig = plt.figure()
        shap.plots.waterfall(e, max_display=14, show=False)
        save_fig(f'interp_10_waterfall_{label}', plt.gcf()); plt.show()
    except Exception as ex:
        print(f'  waterfall skipped ({type(ex).__name__})')

## 9. Cohort analysis — where the spatial signal is used

EXP-006/007 established that spatial features are an *interpolation* aid. Splitting mean
$|\text{SHAP}|$ of the spatial block by station isolation (`nearest_dist`) tests a concrete
prediction of that mechanism: the model should lean on neighbour features more for well-connected
stations and less for isolated ones, where the neighbours are physically uninformative.

In [ ]:
log_section('9. Cohort — spatial reliance vs station isolation')
nd = X_s['nearest_dist'].values
spatial_idx = [FEATURES.index(f) for f in SPATIAL_FEATURES if f != 'nearest_dist']
spatial_abs = np.abs(shap_values[:, spatial_idx]).sum(1)
q = pd.qcut(nd, 4, labels=['nearest','near','far','isolated'])
coh = pd.DataFrame({'isolation': q, 'spatial_|SHAP|': spatial_abs}).groupby('isolation', observed=True)['spatial_|SHAP|'].mean()
print('Mean summed spatial |SHAP| by station-isolation quartile:')
print(coh.round(4).to_string())
fig, ax = plt.subplots(figsize=(6.5,4))
ax.bar(coh.index.astype(str), coh.values, color='crimson', alpha=0.8)
ax.set_ylabel('summed spatial |SHAP|'); ax.set_xlabel('station isolation (nearest neighbour distance)')
ax.set_title('Spatial-feature reliance vs station isolation')
plt.tight_layout(); save_fig('interp_11_cohort_isolation', fig); plt.show()

## 10. Finalization

In [ ]:
log_section('Finalization')
print(f'Results file: {_PATH}')
figs = sorted(f for f in os.listdir('figures') if f.startswith('interp_'))
for f in figs: print(f'  figures/{f}')
models = sorted(f for f in os.listdir(MODEL_DIR) if f.startswith('interpretability_'))
print(f'Models saved ({len(models)}) in {MODEL_DIR}:')
for f in models: print(f'  {f}')
sys.stdout = _ORIG; _FILE.close()
ZIP = 'XGB_interp_artifacts.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(_PATH)
    for f in figs: zf.write(f'figures/{f}')
print(f'Bundled -> {ZIP}')
if IN_COLAB:
    from google.colab import files; files.download(ZIP)
print('Done.')